# MedRAX Premium — Benchmark Evaluation

**Pipeline**: ReAct Agent → Multi-Tool Execution → Canonical Output → Conflict Detection (BERT NLI + Rule-Based + GACL) → Conflict Resolution (Argumentation Graph + Tool Trust + Abstention Logic)

**Benchmark**: ChestAgentBench (MCQ on chest X-rays)

**Environment**: Kaggle 2×T4 GPUs, 4-bit quantization

In [ ]:
# ── Inspect all model directories ──────────────────────────────
import os
from pathlib import Path

_B = Path("/kaggle/input/datasets/mohammadninadmahmud")

MODEL_DIRS = {
    "MODELS_CHEX":    _B / "models-chexagent"     / "models-chexagent",
    "MODELS_CORE":    _B / "models-core"          / "models-core",
    "MODELS_LLAVA":   _B / "models-llava-med"     / "models-llava-med",
    "MODELS_MEDTOOL": _B / "models-medical-tools" / "models-medical-tools",
}

for label, root in MODEL_DIRS.items():
    print(f"\n{'='*70}")
    print(f"  {label}  →  {root}")
    print(f"  exists={root.exists()}")
    print(f"{'='*70}")
    if not root.exists():
        continue
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        indent = "  " * depth
        print(f"{indent}{Path(dirpath).name}/")
        for f in sorted(filenames):
            size_mb = os.path.getsize(os.path.join(dirpath, f)) / 1024**2
            print(f"{indent}  {f}  ({size_mb:.1f} MB)")

In [ ]:
# ===============================
# 📦 Cell 2 — Install Dependencies
# ===============================

# Install lightweight packages (no torch upgrade)
%pip install -q \
    langchain-openai \
    langchain-core \
    langchain-community \
    langgraph \
    tenacity \
    python-dotenv \
    accelerate \
    sentencepiece \
    timm \
    einops \
    shortuuid \
    gradio \
    torchxrayvision \
    scikit-image \
    pydicom

# Install transformers WITHOUT touching torch/torchvision
%pip install -q --no-deps "transformers>=4.40.0" tokenizers huggingface-hub safetensors

# bitsandbytes needs its own install (GPU-specific build)
%pip install -q bitsandbytes

print("✅ All packages installed — no restart needed")


In [ ]:

# ===============================
# 📦 Cell 3 — Core Imports
# ===============================

import operator
import warnings
from typing import *
import traceback
import os
import torch
import json
import glob
import time
import re
import gc
import sys
import uuid
import logging
from datetime import datetime
from pathlib import Path
from collections import defaultdict, Counter

from dotenv import load_dotenv
from IPython.display import Image
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from tenacity import retry, wait_exponential, stop_after_attempt
import matplotlib.pyplot as plt
import numpy as np

# ── Transformers compatibility shim (Kaggle runtime) ──
import transformers.pytorch_utils as _ptu
if not hasattr(_ptu, "find_pruneable_heads_and_indices"):
    def _find_pruneable(heads, n_heads, head_size, already_pruned):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned
        for h in heads:
            h -= sum(1 if ph < h else 0 for ph in already_pruned)
            mask[h] = 0
        return heads, torch.arange(mask.numel())[mask.view(-1).contiguous().eq(1)].long()
    _ptu.find_pruneable_heads_and_indices = _find_pruneable
if not hasattr(_ptu, "prune_linear_layer"):
    import torch.nn as _nn
    def _prune_linear(layer, index, dim=0):
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        b = None
        if layer.bias is not None:
            b = layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach()
        new = _nn.Linear(W.size(1), W.size(0), bias=b is not None).to(layer.weight.device)
        new.weight.requires_grad_(False).copy_(W).requires_grad_(True)
        if b is not None:
            new.bias.requires_grad_(False).copy_(b).requires_grad_(True)
        return new
    _ptu.prune_linear_layer = _prune_linear

# ===============================
# 🔑 API Keys — Kaggle Secrets → os.environ
# ===============================
# On Kaggle: add OPENAI_API_KEY + OPENAI_BASE_URL in the Secrets panel
# (right sidebar) and enable "Notebook access" for both.
# GitHub PATs require base_url="https://models.inference.ai.azure.com"
# load_dotenv() handles local .env files for non-Kaggle runs.

load_dotenv()  # picks up .env locally (no-op on Kaggle)

try:
    from kaggle_secrets import UserSecretsClient as _KSC
    _ks = _KSC()
    for _sname in ["OPENAI_API_KEY", "OPENAI_BASE_URL"]:
        try:
            _sval = _ks.get_secret(_sname)
            if _sval:
                os.environ[_sname] = _sval
                _display = _sval[:8] + "..." if len(_sval) > 8 else "***"
                print(f"  ✅ Kaggle secret loaded: {_sname} = {_display}")
        except Exception:
            pass  # secret not defined in panel — skip silently
except ImportError:
    pass  # not running on Kaggle

# Hard fail early if no key at all — saves confusion later
_api_key = os.environ.get("OPENAI_API_KEY", "")
if not _api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set!\n"
        "  • On Kaggle: add it in the Secrets panel (right sidebar) and enable 'Notebook access'\n"
        "  • Locally: add OPENAI_API_KEY=sk-... to your .env file"
    )

# ── GitHub Models auto-detection ──────────────────────────────────────
# A GitHub PAT (starts with "github_" or "ghp_") only works against the
# GitHub Models inference endpoint — NOT api.openai.com.
# Auto-set OPENAI_BASE_URL if the user forgot to add it as a secret.
_GITHUB_MODELS_URL = "https://models.inference.ai.azure.com"
_is_github_pat = _api_key.startswith(("github_", "ghp_"))
if _is_github_pat and not os.environ.get("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = _GITHUB_MODELS_URL
    print(f"  🔀 GitHub PAT detected → OPENAI_BASE_URL auto-set to {_GITHUB_MODELS_URL}")

_base_url = os.environ.get("OPENAI_BASE_URL", "")
print(f"  ✅ OPENAI_API_KEY ready ({_api_key[:10]}...)")
if _base_url:
    print(f"  ✅ OPENAI_BASE_URL = {_base_url}")
else:
    print(f"  ℹ️  No OPENAI_BASE_URL — using OpenAI directly (api.openai.com)")

# ===============================
# 🔧 Kaggle ROOT + Path Detection
# ===============================

INPUT = Path("/kaggle/input")
WORK  = Path("/kaggle/working")
_B = Path("/kaggle/input/datasets/mohammadninadmahmud")
if not _B.is_dir():
    _B = INPUT

ROOT           = _B / "medrax-premium"       / "Medrax_premium"
BENCH_ROOT     = _B / "chestagentbench"       / "chestagentbench"
MODELS_CHEX    = _B / "models-chexagent"      / "models-chexagent"
MODELS_CORE    = _B / "models-core"           / "models-core"
MODELS_LLAVA   = _B / "models-llava-med"      / "models-llava-med"
MODELS_MEDTOOL = _B / "models-medical-tools"  / "models-medical-tools"

# ── Add ROOT to path BEFORE importing medrax_premium ──
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Import medrax_premium ──
from medrax_premium.agent import *
from medrax_premium.tools import *
from medrax_premium.utils import *

warnings.filterwarnings("ignore")

# ===============================
# 📂 Directory Configuration
# ===============================

PROMPT_FILE    = ROOT / "medrax_premium" / "docs" / "system_prompts.txt"
METADATA_FILE  = BENCH_ROOT / "metadata.jsonl"
FIGURES_DIR    = BENCH_ROOT / "figures"

# Tell agent tools where benchmark images live (for relative path resolution)
os.environ["MEDRAX_FIGURES_DIR"] = str(BENCH_ROOT)

paths = {
    "ROOT": ROOT, "BENCH_ROOT": BENCH_ROOT,
    "MODELS_CHEX": MODELS_CHEX, "MODELS_CORE": MODELS_CORE,
    "MODELS_LLAVA": MODELS_LLAVA, "MODELS_MEDTOOL": MODELS_MEDTOOL,
}
for k, v in paths.items():
    print(f"  {'✅' if v.is_dir() else '❌'} {k} → {v}")

# ===============================
# 🗂️ Logging Setup
# ===============================

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

model_name  = "medrax_premium"
temperature = 0.2
DEVICE_0    = "cuda:0"
DEVICE_1    = "cuda:1"

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
for i in range(torch.cuda.device_count()):
    total = torch.cuda.mem_get_info(i)[1] / 1024**3
    print(f"  GPU-{i}: {torch.cuda.get_device_name(i)} — {total:.1f} GB")

LOG_DIR  = WORK / "medrax_premium_logs"
TEMP_DIR = WORK / "temp"
ACL_DIR  = WORK / "acl_submission"
for d in (LOG_DIR, TEMP_DIR, ACL_DIR):
    d.mkdir(parents=True, exist_ok=True)

log_filename = LOG_DIR / f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

logging.basicConfig(
    filename=str(log_filename),
    level=logging.INFO,
    format="%(message)s",
    force=True,
)


print(f"Logging to: {log_filename}")
print(f"Target layout — GPU-0: report+cls+seg+LLaVA-Med ≈5.5GB | GPU-1: CheXagent ≈5GB (4-bit)")

In [ ]:

# =====================================================================
# 🔧 Cell 3b — GPU Utilities (free-space guards, checkpoint validation,
#              4-bit overflow budget, max_memory pinning)
# =====================================================================

# ---------------------------------------------------------------------------
# GPU helpers — used by the tool-loading cell below
# ---------------------------------------------------------------------------

def parse_gpu_id(device) -> int:
    """Extract integer GPU index from a device string like 'cuda:1'."""
    s = str(device)
    return int(s.split(":")[-1]) if ":" in s else 0


def check_gpu_free_space(device, required_gb: float, label: str = "model") -> float:
    """Raise RuntimeError if target GPU lacks *required_gb* free VRAM.
    Returns current free GB for logging."""
    if not torch.cuda.is_available():
        print(f"  [{label}] CUDA unavailable — skipping VRAM check")
        return 0.0
    gid = parse_gpu_id(device)
    if gid >= torch.cuda.device_count():
        raise RuntimeError(
            f"[{label}] GPU-{gid} does not exist "
            f"(only {torch.cuda.device_count()} GPU(s) visible)"
        )
    free_b, total_b = torch.cuda.mem_get_info(gid)
    free_gb = free_b / 1024**3
    total_gb = total_b / 1024**3
    print(f"  [{label}] GPU-{gid}: {free_gb:.2f}/{total_gb:.2f} GB free (need ~{required_gb:.1f} GB)")
    if free_gb < required_gb:
        raise RuntimeError(
            f"[{label}] GPU-{gid} only {free_gb:.2f} GB free — "
            f"need ~{required_gb:.1f} GB. Free VRAM or enable 4-bit."
        )
    return free_gb


def gpu_summary() -> str:
    """One-liner per GPU: used / total."""
    lines = []
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        used = (total - free) / 1024**3
        total_gb = total / 1024**3
        lines.append(f"  GPU-{i}: {used:.2f}/{total_gb:.2f} GB used")
    return "\n".join(lines)


def validate_checkpoint(path, label: str = "model", required_globs=None):
    """Verify model weight directory exists (+ optional file patterns)."""
    p = Path(path)
    if not p.is_dir():
        raise FileNotFoundError(
            f"[{label}] Checkpoint dir not found: {p}\n"
            f"  Ensure the Kaggle dataset is attached."
        )
    if required_globs:
        for pat in required_globs:
            if not list(p.glob(pat)):
                raise FileNotFoundError(
                    f"[{label}] Required '{pat}' not found in {p}\n"
                    f"  Contents: {sorted(f.name for f in p.iterdir())[:15]}"
                )
    print(f"  [{label}] ✅ {p}")
    return p


def compute_max_memory(device, headroom_gb: float = 1.0, cpu_gb: int = 32):
    """Build max_memory dict that pins quantised model to *device* only.
    Other GPUs get '0GiB' so accelerate never spills there."""
    gid = parse_gpu_id(device)
    mm = {}
    for i in range(torch.cuda.device_count()):
        if i == gid:
            free = torch.cuda.mem_get_info(i)[0] / 1024**3
            budget = max(4, int(free - headroom_gb))
            mm[i] = f"{budget}GiB"
        else:
            mm[i] = "0GiB"
    mm["cpu"] = f"{cpu_gb}GiB"
    return mm


def check_4bit_overflow(device, est_4bit_gb: float, label: str = "model"):
    """Check if 4-bit model fits or needs CPU overflow layers.
    Returns (needs_overflow: bool, free_gb: float)."""
    if not torch.cuda.is_available():
        return True, 0.0
    gid = parse_gpu_id(device)
    free_gb = torch.cuda.mem_get_info(gid)[0] / 1024**3
    scratch = 0.5  # staging / KV-cache scratch
    needs = free_gb < (est_4bit_gb + scratch)
    if needs:
        print(
            f"  [{label}] GPU-{gid}: {free_gb:.2f} GB free but need "
            f"~{est_4bit_gb:.1f}+{scratch} GB → CPU offload for overflow layers"
        )
    else:
        print(
            f"  [{label}] GPU-{gid}: {free_gb:.2f} GB free — "
            f"{est_4bit_gb:.1f} GB 4-bit fits with "
            f"{free_gb - est_4bit_gb:.1f} GB headroom"
        )
    return needs, free_gb


print("✅ GPU utility functions defined")
print(gpu_summary())


In [ ]:
# =====================================================================
# Cell 3c — Pre-download DeBERTa NLI model for BERT conflict detection
# =====================================================================
# The BERT conflict detector uses microsoft/deberta-base-mnli (~550 MB).
# This cell downloads and caches it so the diagnostic check in Cell 4
# passes and full semantic conflict detection is available.
# Only downloads once — subsequent runs use the HuggingFace cache.

import os, logging as _logging
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import transformers as _tf

_nli_model_name = "microsoft/deberta-base-mnli"

# ── Suppress noisy background safetensors-conversion thread ──────────
# Without an HF token the conversion API call fails (non-fatal but
# pollutes output with a long traceback in a background thread).
# DISABLE_SAFETENSORS_CONVERSION=1 stops transformers from spawning it.
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"
# Suppress "unauthenticated requests" warning from huggingface_hub
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
# Silence the DebertaForSequenceClassification LOAD REPORT (not an error)
_tf.logging.set_verbosity_error()

try:
    # Check if both tokenizer AND model weights are already cached
    AutoTokenizer.from_pretrained(_nli_model_name, local_files_only=True)
    AutoModelForSequenceClassification.from_pretrained(
        _nli_model_name, local_files_only=True
    )
    print(f"✅ {_nli_model_name} already cached — nothing to download")
except Exception:
    print(f"⬇️  Downloading {_nli_model_name} (~550 MB) ...")
    AutoTokenizer.from_pretrained(_nli_model_name)
    AutoModelForSequenceClassification.from_pretrained(_nli_model_name)
    print(f"✅ {_nli_model_name} downloaded and cached successfully")

# Restore normal verbosity for subsequent cells
_tf.logging.set_verbosity_warning()


In [ ]:

# =====================================================================
# Cell 4 — Tool Loading (Balanced 2-GPU Layout)
# =====================================================================
#
#   GPU-0 (cuda:0): report (SwinV2x2 fp16)  ~0.5 GB
#                    + classification (DenseNet fp32) ~0.03 GB
#                    + segmentation (PSPNet fp32) ~0.1 GB
#                    + LLaVA-Med 7B (NF4 4-bit) ~4.5 GB
#                    ---------------------------------------- ~5.5 GB
#
#   GPU-1 (cuda:1): CheXagent 8B (NF4 4-bit) ~5 GB
# =====================================================================

from transformers import AutoModelForCausalLM as _AMCLM

# ========== PHASE 0: Validate all model checkpoints ==========
print("=" * 60)
print("Phase 0 -- Validating model checkpoints (Kaggle datasets)")
print("=" * 60)

validate_checkpoint(MODELS_CORE / "swinv2_findings",   label="SwinV2-findings",   required_globs=["*.safetensors", "config.json"])
validate_checkpoint(MODELS_CORE / "swinv2_impression",  label="SwinV2-impression",  required_globs=["*.safetensors", "config.json"])
validate_checkpoint(MODELS_LLAVA,                       label="LLaVA-Med-7B",       required_globs=["config.json"])
validate_checkpoint(MODELS_CHEX,                        label="CheXagent-8B",       required_globs=["config.json"])

print()

# ========== PHASE 1: GPU-0 lightweight tools ====================
print("=" * 60)
print("Phase 1 -- Loading lightweight tools on GPU-0")
print("=" * 60)

check_gpu_free_space(DEVICE_0, required_gb=0.8, label="GPU-0 (lightweight)")
t0 = time.time()

report_tool = ChestXRayReportGeneratorTool(
    cache_dir=str(MODELS_CORE), device=DEVICE_0,
    findings_model_path=str(MODELS_CORE / "swinv2_findings"),
    impression_model_path=str(MODELS_CORE / "swinv2_impression"),
)
xray_classification_tool = ChestXRayClassifierTool(device=DEVICE_0)
segmentation_tool = ChestXRaySegmentationTool(device=DEVICE_0)

free0a, total0 = torch.cuda.mem_get_info(0)
print(f"3 lightweight tools on GPU-0 in {time.time()-t0:.0f}s  "
      f"|  {(total0-free0a)/1024**3:.2f}/{total0/1024**3:.1f} GB")
print()

# ========== PHASE 2: GPU-0 LLaVA-Med 7B (4-bit NF4) ============
print("=" * 60)
print("Phase 2 -- Loading LLaVA-Med 7B on GPU-0 (4-bit NF4)")
print("=" * 60)

_llava_est = 4.5  # estimated 4-bit footprint
check_gpu_free_space(DEVICE_0, required_gb=_llava_est, label="LLaVA-Med-4bit")
needs_overflow_llava, free_before_llava = check_4bit_overflow(
    DEVICE_0, est_4bit_gb=_llava_est, label="LLaVA-Med-4bit",
)

t_llava = time.time()
llava_med_tool = LlavaMedTool(
    model_path=str(MODELS_LLAVA), cache_dir=str(MODELS_LLAVA),
    device=DEVICE_0, load_in_4bit=True,
)

free0b, _ = torch.cuda.mem_get_info(0)
print(f"LLaVA-Med on GPU-0: {(total0-free0b)/1024**3:.2f}/{total0/1024**3:.1f} GB  "
      f"(took {time.time()-t_llava:.0f}s)")
print()

# ========== PHASE 3: GPU-1 CheXagent 8B (4-bit NF4) =============
print("=" * 60)
print("Phase 3 -- Loading CheXagent 8B on GPU-1 (4-bit NF4)")
print("=" * 60)

# Monkey-patch: inject torch_dtype=bf16 so non-quantised layers
# do not materialise in fp32 (~16 GB instead of ~5 GB).
_real_from_pretrained = _AMCLM.from_pretrained

def _patched_from_pretrained(pretrained_model_name_or_path, *args, **kwargs):
    kwargs.setdefault("torch_dtype", torch.bfloat16)
    kwargs.setdefault("low_cpu_mem_usage", True)
    return _real_from_pretrained(pretrained_model_name_or_path, *args, **kwargs)

_AMCLM.from_pretrained = _patched_from_pretrained

_chex_est = 5.0
check_gpu_free_space(DEVICE_1, required_gb=_chex_est, label="CheXagent-4bit")
needs_overflow_chex, free_before_chex = check_4bit_overflow(
    DEVICE_1, est_4bit_gb=_chex_est, label="CheXagent-4bit",
)

t_chex = time.time()
xray_vqa_tool = XRayVQATool(
    model_name=str(MODELS_CHEX), cache_dir=str(MODELS_CHEX),
    device=DEVICE_1, load_in_4bit=True,
)

# Restore original from_pretrained
_AMCLM.from_pretrained = _real_from_pretrained

# ── Hotfix: get_head_mask removed in transformers ≥4.41 ──────────
# CheXagent-8b's QFormer calls self.get_head_mask() which no longer
# exists on PreTrainedModel.  Inject it onto every sub-module that
# has a forward() referencing it.
def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
    if head_mask is not None:
        if head_mask.dim() == 1:
            head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
        elif head_mask.dim() == 2:
            head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
        if is_attention_chunked:
            head_mask = head_mask.unsqueeze(-1)
    else:
        head_mask = [None] * num_hidden_layers
    return head_mask

# Patch onto QFormer (and any other sub-module that needs it)
_chex_model = xray_vqa_tool.model
for _name, _mod in _chex_model.named_modules():
    if hasattr(type(_mod), 'forward') and not hasattr(_mod, 'get_head_mask'):
        # Only patch modules whose forward actually calls get_head_mask
        import inspect as _insp
        try:
            _src = _insp.getsource(type(_mod).forward)
        except (TypeError, OSError):
            _src = ""
        if 'get_head_mask' in _src:
            import types as _types
            _mod.get_head_mask = _types.MethodType(_get_head_mask, _mod)
            print(f"  [Hotfix] Patched get_head_mask onto {_name}")
print("  [Hotfix] get_head_mask patch applied ✅")

free1, total1 = torch.cuda.mem_get_info(1)
print(f"CheXagent on GPU-1: {(total1-free1)/1024**3:.2f}/{total1/1024**3:.1f} GB  "
      f"(took {time.time()-t_chex:.0f}s)")
print()

# ========== PHASE 4b: Monkey-patch BLIP-2 prompt ==================
# CheXagent-8b (BLIP-2 arch) gives very terse output (1-2 findings)
# with bare prompts.  We wrap the user prompt with an enumeration
# instruction so it lists ALL visible pathologies — critical for the
# conflict-resolution pipeline to have enough findings to compare.
# -------------------------------------------------------------------
if xray_vqa_tool._is_blip2:
    _orig_gen_inner = xray_vqa_tool._generate_response_inner

    def _patched_generate_response_inner(
        image_paths, prompt, max_new_tokens,
        do_sample=False, temperature=1.0, top_p=1.0,
    ):
        _enum_prefix = (
            "You are an expert radiologist. Carefully examine this chest X-ray "
            "and provide a detailed, structured analysis. "
            "List ALL visible findings, abnormalities, and relevant negative findings. "
            "For each finding state: (1) the pathology name, (2) its location, "
            "(3) severity (mild/moderate/severe), and (4) your confidence. "
            "Do not summarize — enumerate every observation.\n\n"
        )
        return _orig_gen_inner(
            image_paths, _enum_prefix + prompt, max_new_tokens,
            do_sample=do_sample, temperature=temperature, top_p=top_p,
        )

    xray_vqa_tool._generate_response_inner = _patched_generate_response_inner
    print("  [Patch] BLIP-2 prompt enhanced for detailed pathology enumeration ✅")
else:
    print("  [Info] Qwen-VL arch — no BLIP-2 prompt patch needed")

# ========== PHASE 4c: Fix TOOL_EXPERTISE tool names ==================
# ConflictResolver.TOOL_EXPERTISE uses wrong names for localization.
# Actual tool names: "chest_xray_segmentation", "xray_phrase_grounding"
from medrax_premium.agent.conflict_resolution import ConflictResolver
ConflictResolver.TOOL_EXPERTISE["localization"] = {
    "primary": "chest_xray_segmentation",
    "fallback": "xray_phrase_grounding",
    "description": "Precise anatomical location",
}
print("  [Patch] ConflictResolver.TOOL_EXPERTISE localization names fixed ✅")

# ========== PHASE 5: Cleanup + Summary =============================
gc.collect(); torch.cuda.empty_cache()

ALL_TOOLS = [
    report_tool, xray_classification_tool, segmentation_tool,
    llava_med_tool, xray_vqa_tool,
]

free0_final, _ = torch.cuda.mem_get_info(0)
free1_final, _ = torch.cuda.mem_get_info(1)
print("=" * 60)
print(f"{len(ALL_TOOLS)} tools loaded -- balanced 2-GPU layout")
print(f"   GPU-0 (report+cls+seg+LLaVA): "
      f"{(total0-free0_final)/1024**3:.2f}/{total0/1024**3:.1f} GB")
print(f"   GPU-1 (CheXagent):             "
      f"{(total1-free1_final)/1024**3:.2f}/{total1/1024**3:.1f} GB")
if needs_overflow_llava:
    print("   LLaVA-Med used CPU offload for some layers")
if needs_overflow_chex:
    print("   CheXagent used CPU offload for some layers")
print("=" * 60)

# ========== DIAGNOSTIC: BERT NLI availability check =================
# DeBERTa needs to be either cached locally or internet-downloadable.
# On Kaggle without internet, it will silently fall back to rule-based.
_bert_available = False
try:
    from transformers import AutoTokenizer
    _test_tok = AutoTokenizer.from_pretrained(
        "microsoft/deberta-base-mnli",
        local_files_only=True,  # Don't download — just check cache
    )
    _bert_available = True
    del _test_tok
except Exception:
    pass

if _bert_available:
    print("✅ BERT NLI model (deberta-base-mnli) found in cache — full semantic conflict detection")
else:
    print("⚠️  BERT NLI model NOT cached — will use rule-based conflict detection only")
    print("   (Rule-based still catches presence/absence conflicts; GACL catches anatomical inconsistencies)")
    print("   To enable BERT: pre-download deberta-base-mnli or add internet access")


# =====================================================================
# Agent + Runner
# =====================================================================

def get_agent():
    """Create a fresh Premium agent with conflict resolution enabled."""
    prompts = load_prompts_from_file(str(PROMPT_FILE))
    prompt = prompts["MEDICAL_ASSISTANT"]

    # Resolve API key — Cell 3 already guaranteed this is set;
    # we retrieve it again here in case get_agent() is called standalone.
    _key = os.environ.get("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not _key:
        raise EnvironmentError(
            "OPENAI_API_KEY missing. Run Cell 3 first, or set the Kaggle Secret."
        )

    model = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=temperature,
        top_p=0.95,
        api_key=_key,
        base_url=os.environ.get("OPENAI_BASE_URL") or None,
    )
    agent = Agent(
        model, tools=ALL_TOOLS, log_tools=True,
        log_dir=str(LOG_DIR / "tool_logs"),
        system_prompt=prompt, checkpointer=MemorySaver(),
        enable_conflict_resolution=True,
        conflict_sensitivity=0.25, deferral_threshold=0.6,
    )
    # Pin BERT conflict detector to CPU to avoid GPU-0 OOM (DeBERTa ~400MB)
    if hasattr(agent, "conflict_detector") and agent.conflict_detector.use_bert:
        agent.conflict_detector.bert_device = "cpu"
    thread = {"configurable": {"thread_id": str(uuid.uuid4())}}
    return agent, thread


def run_medrax(agent, thread, prompt, image_urls=[], _max_retries=5):
    """Run a single turn through the ReAct agent with rate-limit retry."""
    messages = [
        HumanMessage(
            content=[{"type": "text", "text": prompt}]
            + [{"type": "image_url", "image_url": {"url": url}} for url in image_urls]
        )
    ]

    for attempt in range(_max_retries):
        try:
            final_response = None
            for event in agent.workflow.stream({"messages": messages}, thread):
                for v in event.values():
                    final_response = v

            if isinstance(final_response, dict) and final_response.get("messages"):
                final_response = final_response["messages"][-1].content.strip()
            elif final_response is not None:
                final_response = str(final_response).strip()

            agent_state = agent.workflow.get_state(thread)
            return final_response or "", str(agent_state)

        except Exception as e:
            err_str = str(e)

            # Retry on rate-limit (429) with exponential backoff
            if "429" in err_str or "RateLimitReached" in err_str or "rate limit" in err_str.lower():
                wait = min(2 ** attempt * 30, 300)  # 30s, 60s, 120s, 240s, 300s
                print(f"  ⏳ Rate limited (attempt {attempt+1}/{_max_retries}), waiting {wait}s...")
                time.sleep(wait)
                continue

            raise  # Non-rate-limit errors bubble up immediately

    raise RuntimeError(f"Rate limit exceeded after {_max_retries} retries")

In [ ]:

# ===============================
# 📝 Cell 5 — Question Processing + Main
# ===============================
import base64, mimetypes


def _image_to_data_uri(path):
    """Convert a local image file to a data URI for the LLM."""
    mime, _ = mimetypes.guess_type(str(path))
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return f"data:{mime or 'image/jpeg'};base64,{b64}"


def _extract_letter(text):
    """Extract a single answer letter from model output."""
    text = text.strip()
    if len(text) == 1 and text.upper() in "ABCDEF":
        return text.upper()
    m = re.match(r'^([A-Fa-f])\b', text)
    if m: return m.group(1).upper()
    m = re.search(r'(?:answer\s+is|option|choose)\s+([A-Fa-f])\b', text, re.I)
    if m: return m.group(1).upper()
    m = re.search(r'\(([A-Fa-f])\)', text)
    if m: return m.group(1).upper()
    m = re.search(r'\b([A-Fa-f])\b', text)
    if m: return m.group(1).upper()
    return text.strip()


def create_multimodal_request(question_data, case_id, question_id, agent, thread):
    """Two-turn ReAct evaluation: reasoning + force letter. Mirrors original MedRAX."""

    # ── Resolve images (ChestAgentBench format) ──
    local_images = question_data.get("images", [])
    if isinstance(local_images, str):
        local_images = [local_images]

    image_urls = []
    image_paths = []
    figure_prompt = ""
    for img in local_images:
        fp = BENCH_ROOT / img
        if fp.exists():
            image_urls.append(_image_to_data_uri(fp))
            image_paths.append(str(fp))
            figure_prompt += f"  Image path for tools: {fp}\n"

    if not image_urls:
        print(f"  ⏭ No images found — skipping")
        return None, ""

    prompt = (
        "Answer this question correctly using chain of thought reasoning and "
        "carefully evaluating choices. Solve using your own vision and reasoning and then "
        "use tools to complement your reasoning. Trust your own judgement over any tools.\n"
        "IMPORTANT: For thorough analysis, always use at least TWO diagnostic tools "
        "(e.g. chest_xray_classifier AND chest_xray_expert) on the image before answering. "
        "This enables cross-validation of findings.\n"
        f"{question_data['question']}\n{figure_prompt}"
    )

    try:
        start_time = time.time()

        # Turn 1: reasoning + tool calls (ReAct loop)
        final_response, agent_state = run_medrax(
            agent=agent, thread=thread, prompt=prompt, image_urls=image_urls,
        )

        # Turn 2: force single letter answer
        model_answer, agent_state = run_medrax(
            agent=agent,
            thread=thread,
            prompt="If you had to choose the best option, only respond with the letter of choice (only one of A, B, C, D, E, F)",
        )
        duration = time.time() - start_time

        answer = _extract_letter(model_answer)
        correct = question_data.get("answer", "").strip().upper()

        log_entry = {
            "case_id": case_id,
            "question_id": question_id,
            "timestamp": datetime.now().isoformat(),
            "model": model_name,
            "temperature": temperature,
            "duration": round(duration, 2),
            "usage": "",
            "cost": 0,
            "raw_response": final_response[:2000],
            "model_answer": answer,
            "correct_answer": correct,
            "is_correct": answer == correct,
            "categories": question_data.get("categories", ""),
            "type": question_data.get("type", ""),
            "input": {
                "messages": prompt[:1000],
                "question_data": {
                    "question": question_data["question"][:1000],
                    "figures": question_data.get("images", []),
                },
                "image_paths": image_paths,
            },
            "agent_state": agent_state[:2000],
        }
        logging.info(json.dumps(log_entry, default=str))
        return final_response, answer

    except Exception as e:
        log_entry = {
            "case_id": case_id,
            "question_id": question_id,
            "timestamp": datetime.now().isoformat(),
            "model": model_name,
            "temperature": temperature,
            "status": "error",
            "error": str(e),
            "cost": 0,
            "input": {
                "messages": prompt[:1000],
                "question_data": {
                    "question": question_data.get("question", "")[:1000],
                    "figures": question_data.get("images", []),
                },
                "image_paths": image_paths,
            },
        }
        logging.info(json.dumps(log_entry, default=str))
        traceback.print_exc()
        print(f"Error processing case {case_id}, question {question_id}: {e}")
        return "", ""


def load_benchmark_questions():
    """Load ChestAgentBench questions from metadata.jsonl."""
    with open(METADATA_FILE) as f:
        return [json.loads(line.strip()) for line in f]


PREMIUM_LOG = LOG_DIR / "premium_benchmark.jsonl"
MAX_QUESTIONS = 5


def main():
    """Run benchmark evaluation — resumable via PREMIUM_LOG."""
    all_questions = load_benchmark_questions()

    # Group by case
    cases = defaultdict(list)
    for q in all_questions:
        cases[q.get("case_id", "unknown")].append(q)
    case_ids = list(cases.keys())

    total_cases, total_questions = len(case_ids), len(all_questions)

    # Resume logic
    completed_qids = set()
    prev_correct = prev_done = prev_skipped = prev_errors = 0
    if PREMIUM_LOG.exists():
        with open(PREMIUM_LOG) as f:
            for line in f:
                try:
                    entry = json.loads(line.strip())
                    qid = entry.get("question_id")
                    if qid:
                        completed_qids.add(qid)
                    st = entry.get("status", "")
                    if st == "skipped":
                        prev_skipped += 1
                    elif st == "error":
                        prev_errors += 1
                    elif "model_answer" in entry:
                        prev_done += 1
                        if entry.get("is_correct", False):
                            prev_correct += 1
                except json.JSONDecodeError:
                    continue
        print(f"🔄 RESUMING — {len(completed_qids)} already done (acc={prev_correct}/{prev_done})")
    else:
        print(f"🆕 Starting fresh → {PREMIUM_LOG.name}")

    cases_processed = 0
    questions_processed = len(completed_qids)
    skipped_questions = prev_skipped
    errors = prev_errors
    done = prev_done
    correct = prev_correct
    remaining = max(0, (MAX_QUESTIONS or total_questions) - questions_processed)

    print(f"Beginning benchmark evaluation for model {model_name} with temperature {temperature}")
    print(f"  {total_questions} total Qs | MAX={MAX_QUESTIONS} | remaining={remaining}\n")

    for case_id in case_ids:
        if MAX_QUESTIONS is not None and questions_processed >= MAX_QUESTIONS:
            break

        print(f"----------------------------------------------------------------")
        agent, thread = get_agent()

        question_list = cases[case_id]
        if not question_list:
            continue

        cases_processed += 1
        for q in question_list:
            if MAX_QUESTIONS is not None and questions_processed >= MAX_QUESTIONS:
                break

            qid = q.get("question_id", "unknown")
            if qid in completed_qids:
                continue

            questions_processed += 1
            question_id = qid

            final_response, model_answer = create_multimodal_request(
                q, case_id, question_id, agent, thread,
            )

            # Persist to JSONL immediately (for resume)
            if final_response is None:
                skipped_questions += 1
                entry = {"question_id": qid, "case_id": case_id, "status": "skipped",
                         "timestamp": datetime.now().isoformat()}
                with open(PREMIUM_LOG, "a") as f:
                    f.write(json.dumps(entry, default=str) + "\n")
                print(f"Skipped question: Case={case_id}, QID={question_id}")
                continue

            if final_response == "":
                errors += 1
                entry = {"question_id": qid, "case_id": case_id, "status": "error",
                         "timestamp": datetime.now().isoformat()}
                with open(PREMIUM_LOG, "a") as f:
                    f.write(json.dumps(entry, default=str) + "\n")
                continue

            done += 1
            correct_answer = q.get("answer", "").strip().upper()
            is_correct = model_answer.upper() == correct_answer
            if is_correct:
                correct += 1

            entry = {
                "question_id": qid, "case_id": case_id,
                "status": "processed",
                "timestamp": datetime.now().isoformat(),
                "model_answer": model_answer,
                "correct_answer": correct_answer,
                "is_correct": is_correct,
                "categories": q.get("categories", ""),
                "raw_response": final_response[:2000],
            }
            with open(PREMIUM_LOG, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")

            mark = "✅" if is_correct else "❌"
            acc_pct = 100 * correct / done if done > 0 else 0
            print(
                f"Progress: Case {cases_processed}/{total_cases}, "
                f"Question {questions_processed}/{MAX_QUESTIONS or total_questions}"
            )
            print(f"Case ID: {case_id}")
            print(f"Question ID: {question_id}")
            print(f"Model Answer: {model_answer}")
            print(f"Correct Answer: {correct_answer}")
            print(f"{mark} Running: {correct}/{done} = {acc_pct:.1f}%")
            print(f"----------------------------------------------------------------\n")

        del agent
        gc.collect()

    print(f"\nBenchmark Summary:")
    print(f"Total Cases Processed: {cases_processed}")
    print(f"Total Questions Processed: {questions_processed}")
    print(f"Total Questions Skipped: {skipped_questions}")
    print(f"Total Errors: {errors}")
    if done > 0:
        print(f"ACCURACY: {correct}/{done} = {100*correct/done:.1f}%")

main()


In [ ]:
# ===============================
# 📊 Cell 6 — Results: Accuracy Comparison + Figures (ACL Standard)
# ===============================
import matplotlib
%matplotlib inline

BASELINES = {
    "LLaVA-Med":     {"Detection": 32.4, "Classification": 30.8, "Localization": 30.2,
                      "Comparison": 30.6, "Relationship": 31.8, "Diagnosis": 29.3,
                      "Characterization": 28.8, "Overall": 28.7},
    "CheXagent":     {"Detection": 38.7, "Classification": 34.7, "Localization": 42.5,
                      "Comparison": 38.5, "Relationship": 39.8, "Diagnosis": 33.5,
                      "Characterization": 34.2, "Overall": 39.5},
    "Llama-3.2-90B": {"Detection": 58.1, "Classification": 56.5, "Localization": 59.9,
                      "Comparison": 57.5, "Relationship": 59.3, "Diagnosis": 55.9,
                      "Characterization": 58.0, "Overall": 57.9},
    "GPT-4o":        {"Detection": 58.7, "Classification": 54.6, "Localization": 59.0,
                      "Comparison": 55.5, "Relationship": 59.0, "Diagnosis": 52.6,
                      "Characterization": 56.1, "Overall": 56.4},
    "MedRAX":        {"Detection": 64.1, "Classification": 62.9, "Localization": 63.6,
                      "Comparison": 61.8, "Relationship": 63.1, "Diagnosis": 62.5,
                      "Characterization": 64.0, "Overall": 63.1},
}
CATEGORY_ORDER = ["Detection", "Classification", "Localization", "Comparison",
                  "Relationship", "Diagnosis", "Characterization", "Enumeration", "Reasoning"]

# Load results
results_premium = []
with open(PREMIUM_LOG) as f:
    for line in f:
        results_premium.append(json.loads(line.strip()))

answered = [r for r in results_premium if "model_answer" in r]
for r in answered:
    r["is_correct"] = r["model_answer"].upper().strip() == r["correct_answer"].upper().strip()

total_ans = len(answered)
total_correct = sum(r["is_correct"] for r in answered)

# Per-category accuracy
cat_stats = {c: {"correct": 0, "total": 0} for c in CATEGORY_ORDER}
for r in answered:
    raw = r.get("categories", "")
    cats = [c.strip().capitalize() for c in (raw if isinstance(raw, list) else str(raw).split(",")) if c.strip()]
    for c in cats:
        if c in cat_stats:
            cat_stats[c]["total"] += 1
            if r["is_correct"]:
                cat_stats[c]["correct"] += 1

premium = {}
for cat in CATEGORY_ORDER:
    s = cat_stats[cat]
    premium[cat] = round(100 * s["correct"] / s["total"], 1) if s["total"] > 0 else None
premium["Overall"] = round(100 * total_correct / total_ans, 1) if total_ans > 0 else None

# Print comparison table
model_names = list(BASELINES.keys()) + ["MedRAX Premium"]
print("=" * 110)
print("  ChestAgentBench — Accuracy by Category (%)")
print("=" * 110)
hdr = f"  {'Category':<18s}" + "".join(f"  {m:>14s}" for m in model_names)
print(hdr)
print("  " + "─" * 105)
for cat in CATEGORY_ORDER:
    row = f"  {cat:<18s}"
    for m in model_names:
        val = premium.get(cat) if m == "MedRAX Premium" else BASELINES[m].get(cat)
        row += f"  {val:>14.1f}" if val is not None else f"  {'—':>14s}"
    print(row)
print("  " + "─" * 105)
row = f"  {'Overall':<18s}"
for m in model_names:
    val = premium.get("Overall") if m == "MedRAX Premium" else BASELINES[m].get("Overall", 0)
    row += f"  {val:>14.1f}" if val is not None else f"  {'—':>14s}"
print(row)
print("=" * 110)

# Save JSON
acc_json = {"per_category": {}, "overall": {}, "metadata": {
    "answered": total_ans, "model": model_name, "timestamp": datetime.now().isoformat()}}
for cat in CATEGORY_ORDER:
    acc_json["per_category"][cat] = {
        m: (premium.get(cat) if m == "MedRAX Premium" else BASELINES[m].get(cat))
        for m in model_names
    }
for m in model_names:
    acc_json["overall"][m] = premium.get("Overall") if m == "MedRAX Premium" else BASELINES[m].get("Overall")
with open(ACL_DIR / "table_accuracy_comparison.json", "w") as f:
    json.dump(acc_json, f, indent=2, default=str)
with open(ACL_DIR / "raw_results.json", "w") as f:
    json.dump(results_premium, f, indent=2, default=str)

# Figures
cats_plot = [c for c in CATEGORY_ORDER if premium.get(c) is not None]
if cats_plot:
    x = np.arange(len(cats_plot))
    n_m = len(model_names); w = 0.8 / n_m
    colors = ["#a0a0a0", "#b0b0b0", "#4a90d9", "#f5a623", "#2ecc71", "#e74c3c"]

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, m in enumerate(model_names):
        vals = [premium.get(c, 0) or 0 for c in cats_plot] if m == "MedRAX Premium" \
               else [BASELINES[m].get(c, 0) or 0 for c in cats_plot]
        ax.bar(x + i*w - (n_m-1)*w/2, vals, w, label=m, color=colors[i % len(colors)], edgecolor="white")
    ax.set_ylabel("Accuracy (%)"); ax.set_title("ChestAgentBench — All Models")
    ax.set_xticks(x); ax.set_xticklabels(cats_plot, rotation=25, ha="right")
    ax.set_ylim(0, 105); ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    for ext in ["pdf", "png"]:
        fig.savefig(str(ACL_DIR / f"fig_category_comparison.{ext}"), dpi=300, bbox_inches="tight")
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(10, 4))
    ov = [premium.get("Overall", 0) or 0 if m == "MedRAX Premium" else BASELINES[m].get("Overall", 0) for m in model_names]
    bars = ax2.barh(model_names[::-1], ov[::-1], color=colors[:len(model_names)][::-1], edgecolor="white")
    ax2.set_xlabel("Overall Accuracy (%)"); ax2.set_xlim(0, 100)
    for bar, val in zip(bars, ov[::-1]):
        ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%", va="center", fontweight="bold")
    ax2.grid(axis="x", alpha=0.3); plt.tight_layout()
    for ext in ["pdf", "png"]:
        fig2.savefig(str(ACL_DIR / f"fig_overall_accuracy.{ext}"), dpi=300, bbox_inches="tight")
    plt.show()

print("✅ Accuracy tables & figures saved to", ACL_DIR)

In [ ]:
# ===============================
# 📋 Cell 6b — Detailed Cross-Verification & Paper-Style Tables
# ===============================
# Reproduces the paper table format (image attached), adds per-question
# cross-verification, error analysis, confusion matrix, delta comparison.

import pandas as pd
from IPython.display import display, HTML

# ─────────────────────────────────────────────────────────────────────
# TABLE A — Paper-Style Accuracy Table (matches the attached image)
# ─────────────────────────────────────────────────────────────────────
PAPER_CATEGORIES = ["Detection", "Classification", "Localization",
                    "Comparison", "Relationship", "Diagnosis", "Characterization"]

paper_data = {}
for m, scores in BASELINES.items():
    paper_data[m] = [scores.get(c, None) for c in PAPER_CATEGORIES] + [scores.get("Overall")]
paper_data["MedRAX Premium"] = [premium.get(c, None) for c in PAPER_CATEGORIES] + [premium.get("Overall")]

df_paper = pd.DataFrame(paper_data, index=PAPER_CATEGORIES + ["Overall"])
df_paper.index.name = "Categories"

def highlight_top2(row):
    """Bold best, underline second-best in each row."""
    styles = [""] * len(row)
    valid = sorted(row.dropna().values, reverse=True)
    if len(valid) >= 1:
        best = valid[0]
        second = valid[1] if len(valid) >= 2 else None
        for i, v in enumerate(row):
            if pd.notna(v) and v == best:
                styles[i] = "font-weight: bold"
            elif second is not None and pd.notna(v) and v == second:
                styles[i] = "text-decoration: underline"
    return styles

print("=" * 100)
print("  TABLE A — Paper-Style Accuracy by Category (%)")
print("  (Bold = best, Underline = second-best)")
print("=" * 100)

styled = (df_paper
    .style
    .apply(highlight_top2, axis=1)
    .format("{:.1f}", na_rep="—")
    .set_caption("ChestAgentBench — Accuracy by Category (%)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
        {"selector": "th", "props": [("text-align", "center"), ("padding", "6px")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "6px")]},
    ]))
display(styled)

# ─────────────────────────────────────────────────────────────────────
# TABLE B — Per-Question Cross-Verification
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 100)
print("  TABLE B — Per-Question Cross-Verification")
print("=" * 100)

rows_verify = []
for r in answered:
    cats_raw = r.get("categories", "")
    cats_list = [c.strip() for c in (cats_raw if isinstance(cats_raw, list)
                 else str(cats_raw).split(",")) if c.strip()]
    rows_verify.append({
        "Case ID": r.get("case_id", "?"),
        "Question ID": str(r.get("question_id", "?"))[-12:],
        "Categories": ", ".join(cats_list),
        "Model Ans": r.get("model_answer", "?").upper(),
        "Correct Ans": r.get("correct_answer", "?").upper(),
        "Match": "✅" if r["is_correct"] else "❌",
    })

df_verify = pd.DataFrame(rows_verify)

def color_match(val):
    if val == "✅":
        return "background-color: #d4edda; color: #155724"
    elif val == "❌":
        return "background-color: #f8d7da; color: #721c24"
    return ""

if len(df_verify) > 0 and "Match" in df_verify.columns:
    styled_verify = (df_verify
        .style
        .map(color_match, subset=["Match"])
        .set_caption(f"Per-Question Results — {len(answered)} answered, "
                     f"{total_correct} correct ({100*total_correct/max(total_ans,1):.1f}%)")
        .set_table_styles([
            {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("padding", "4px 8px")]},
        ]))
    display(styled_verify)
else:
    print(f"  No answered questions to display (0/{total_ans} answered).")

# ─────────────────────────────────────────────────────────────────────
# TABLE C — Delta Comparison: MedRAX Premium vs All Baselines
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 100)
print("  TABLE C — Accuracy Delta: MedRAX Premium vs Baselines (pp)")
print("=" * 100)

delta_data = {}
for m in BASELINES:
    deltas = []
    for c in PAPER_CATEGORIES + ["Overall"]:
        p = premium.get(c)
        b = BASELINES[m].get(c)
        if p is not None and b is not None:
            deltas.append(round(p - b, 1))
        else:
            deltas.append(None)
    delta_data[f"vs {m}"] = deltas

df_delta = pd.DataFrame(delta_data, index=PAPER_CATEGORIES + ["Overall"])
df_delta.index.name = "Categories"

def color_delta(val):
    if pd.isna(val):
        return ""
    if val > 0:
        return "color: #27ae60; font-weight: bold"
    elif val < 0:
        return "color: #e74c3c; font-weight: bold"
    return "color: #888"

styled_delta = (df_delta
    .style
    .map(color_delta)
    .format(lambda v: f"+{v:.1f}" if pd.notna(v) and v > 0 else (f"{v:.1f}" if pd.notna(v) else "—"))
    .set_caption("Accuracy Delta: MedRAX Premium − Baseline (percentage points)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold")]},
        {"selector": "td", "props": [("text-align", "center"), ("padding", "5px")]},
    ]))
display(styled_delta)

# ─────────────────────────────────────────────────────────────────────
# TABLE D — Error Analysis: Wrong Answers by Category
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 100)
print("  TABLE D — Error Analysis")
print("=" * 100)

wrong = [r for r in answered if not r["is_correct"]]
error_cats = Counter()
for r in wrong:
    cats_raw = r.get("categories", "")
    cats_list = [c.strip().capitalize() for c in (cats_raw if isinstance(cats_raw, list)
                 else str(cats_raw).split(",")) if c.strip()]
    for c in cats_list:
        error_cats[c] += 1

error_rows = []
for cat in PAPER_CATEGORIES + ["Enumeration", "Reasoning"]:
    s = cat_stats.get(cat, {"correct": 0, "total": 0})
    err = error_cats.get(cat, 0)
    if s["total"] > 0:
        error_rows.append({
            "Category": cat,
            "Total Qs": s["total"],
            "Correct": s["correct"],
            "Wrong": err,
            "Accuracy (%)": round(100 * s["correct"] / s["total"], 1),
            "Error Rate (%)": round(100 * err / s["total"], 1),
        })

if error_rows:
    # Overall row
    error_rows.append({
        "Category": "OVERALL",
        "Total Qs": total_ans,
        "Correct": total_correct,
        "Wrong": total_ans - total_correct,
        "Accuracy (%)": round(100 * total_correct / max(total_ans, 1), 1),
        "Error Rate (%)": round(100 * (total_ans - total_correct) / max(total_ans, 1), 1),
    })

    df_error = pd.DataFrame(error_rows)

    def highlight_overall(row):
        if row["Category"] == "OVERALL":
            return ["font-weight: bold; border-top: 2px solid black"] * len(row)
        return [""] * len(row)

    styled_err = (df_error
        .style
        .apply(highlight_overall, axis=1)
        .background_gradient(subset=["Accuracy (%)"], cmap="RdYlGn", vmin=0, vmax=100)
        .format({"Accuracy (%)": "{:.1f}", "Error Rate (%)": "{:.1f}"})
        .set_caption(f"Error Analysis — {len(wrong)} wrong out of {total_ans}")
        .hide(axis="index"))
    display(styled_err)
else:
    print("  No errors to analyze (or no data yet).")

# ─────────────────────────────────────────────────────────────────────
# TABLE E — Answer Distribution Confusion Matrix
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 100)
print("  TABLE E — Answer Confusion Matrix (Model vs Correct)")
print("=" * 100)

letters = sorted(set(
    [r.get("model_answer", "?").upper() for r in answered] +
    [r.get("correct_answer", "?").upper() for r in answered]
))
# Keep only A-F
letters = [l for l in letters if len(l) == 1 and l in "ABCDEF"]

conf = {c: {m: 0 for m in letters} for c in letters}
for r in answered:
    ma = r.get("model_answer", "?").upper().strip()
    ca = r.get("correct_answer", "?").upper().strip()
    if ca in conf and ma in conf[ca]:
        conf[ca][ma] += 1

df_conf = pd.DataFrame(conf).T
df_conf.index.name = "Correct ↓ / Model →"

def highlight_diagonal(df):
    """Green for diagonal (correct answers), light red for off-diagonal."""
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for i, idx in enumerate(df.index):
        for j, col in enumerate(df.columns):
            if idx == col and df.iloc[i, j] > 0:
                styles.iloc[i, j] = "background-color: #d4edda; font-weight: bold"
            elif df.iloc[i, j] > 0:
                styles.iloc[i, j] = "background-color: #f8d7da"
    return styles

if len(letters) > 0 and total_ans > 0:
    styled_conf = (df_conf
        .style
        .apply(highlight_diagonal, axis=None)
        .set_caption("Answer Confusion Matrix (diagonal = correct)")
        .set_table_styles([
            {"selector": "td", "props": [("text-align", "center"), ("min-width", "40px")]},
        ]))
    display(styled_conf)
else:
    print(f"  Insufficient data for confusion matrix ({len(answered)} answered questions).")

# ─────────────────────────────────────────────────────────────────────
# TABLE F — Summary Statistics
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("  TABLE F — Summary Statistics")
print("=" * 80)

skipped = len([r for r in results_premium if r.get("status") == "skipped"])
errors_count = len([r for r in results_premium if r.get("status") == "error"])

summary_rows = [
    {"Metric": "Total Questions in Log", "Value": len(results_premium)},
    {"Metric": "Answered (with model output)", "Value": total_ans},
    {"Metric": "Correct Answers", "Value": total_correct},
    {"Metric": "Wrong Answers", "Value": total_ans - total_correct},
    {"Metric": "Skipped (no images)", "Value": skipped},
    {"Metric": "Errors (tool/API failures)", "Value": errors_count},
    {"Metric": "Overall Accuracy (%)", "Value": f"{100*total_correct/max(total_ans,1):.1f}%"},
    {"Metric": "Categories Covered", "Value": sum(1 for c in PAPER_CATEGORIES if cat_stats.get(c, {}).get("total", 0) > 0)},
]

df_summary = pd.DataFrame(summary_rows)
styled_summary = (df_summary
    .style
    .set_caption("Benchmark Run Summary")
    .hide(axis="index")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold")]},
        {"selector": "td:nth-child(2)", "props": [("text-align", "right"), ("font-weight", "bold")]},
    ]))
display(styled_summary)

# ─────────────────────────────────────────────────────────────────────
# Save cross-verification as JSON + CSV
# ─────────────────────────────────────────────────────────────────────
df_verify.to_csv(str(ACL_DIR / "cross_verification.csv"), index=False)
df_paper.to_csv(str(ACL_DIR / "paper_table_accuracy.csv"))
if error_rows:
    df_error.to_csv(str(ACL_DIR / "error_analysis.csv"), index=False)

with open(ACL_DIR / "cross_verification_detail.json", "w") as f:
    json.dump({
        "per_question": rows_verify,
        "error_analysis": error_rows if error_rows else [],
        "confusion_matrix": {k: dict(v) for k, v in conf.items()} if letters else {},
        "summary": summary_rows,
    }, f, indent=2, default=str)

print(f"\n✅ Cross-verification tables saved to {ACL_DIR}")
print(f"   • cross_verification.csv  ({len(rows_verify)} rows)")
print(f"   • paper_table_accuracy.csv")
print(f"   • cross_verification_detail.json")
print(f"   • error_analysis.csv")

In [ ]:
# ===============================
# 🔍 Cell 7 — Conflict Resolution Dashboard
# ===============================
# Compact: one 5×5 heatmap matrix + one combined summary table.

import pandas as pd
from IPython.display import display, HTML

# ── All 5 tool names (always show full matrix, even if 0 conflicts) ──
CANONICAL_TOOLS = [
    "chest_xray_report_generator",
    "chest_xray_classifier",
    "chest_xray_segmentation",
    "llava_med_qa",
    "chest_xray_expert",
]
SHORT_NAMES = {
    "chest_xray_report_generator": "ReportGen",
    "chest_xray_classifier":      "Classifier",
    "chest_xray_segmentation":    "Segment",
    "llava_med_qa":               "LLaVA-Med",
    "chest_xray_expert":          "CheXagent",
}

# ── Parse conflict-resolution logs ──
tool_logs_dir = LOG_DIR / "tool_logs"
log_files = sorted(glob.glob(str(tool_logs_dir / "conflict_resolution_*.json")))
print(f"Found {len(log_files)} conflict-resolution log files\n")

det_counts  = Counter()   # pair → total detected
res_counts  = Counter()   # pair → total resolved
severity_counts = Counter()
abstain_by_reason = Counter()
cycle_count = human_review_count = 0

tool_conflict_matrix  = Counter()    # (toolA, toolB) → count
tool_conflict_by_type = {}           # (toolA, toolB) → {type: count}
tool_conflict_severity = {}          # (toolA, toolB) → {severity: count}
tool_conflict_pathology = Counter()  # (toolA, toolB, pathology) → count
all_tool_names = set()

for lf in log_files:
    with open(lf) as f:
        try:
            data = json.load(f)
        except Exception:
            continue
    conflicts   = data.get("conflict_analysis", {}).get("conflicts", [])
    resolutions = data.get("conflict_analysis", {}).get("resolutions", [])

    for i, c in enumerate(conflicts):
        ctype     = c.get("conflict_type", "unknown")
        tools_inv = c.get("tools_involved", [])
        pair      = " vs ".join(sorted(tools_inv)) if tools_inv else "unknown"
        sev       = c.get("severity", "unknown")

        det_counts[pair] += 1
        severity_counts[sev] += 1

        if i < len(resolutions):
            res_counts[pair] += 1
            r = resolutions[i]
            if r.get("argumentation_graph", {}).get("has_cycles", False):
                cycle_count += 1
            if r.get("abstention_reason"):
                abstain_by_reason[r["abstention_reason"]] += 1
            if r.get("should_defer", False):
                human_review_count += 1

        # Build tool-vs-tool data
        if len(tools_inv) >= 2:
            for t in tools_inv:
                all_tool_names.add(t)
            t_pair = tuple(sorted(tools_inv[:2]))
            tool_conflict_matrix[t_pair] += 1
            tool_conflict_by_type.setdefault(t_pair, Counter())[ctype] += 1
            tool_conflict_severity.setdefault(t_pair, Counter())[sev] += 1
            finding = c.get("finding", "unknown")
            tool_conflict_pathology[(t_pair[0], t_pair[1], finding)] += 1

# ═══════════════════════════════════════════════════════════════════
# TABLE 1 — 5×5 TOOL CONFLICT HEATMAP MATRIX
# ═══════════════════════════════════════════════════════════════════
print("=" * 90)
print("  TOOL vs TOOL — 5×5 CONFLICT HEATMAP")
print("=" * 90)

# Always use all 5 tools (merge any extras from logs)
matrix_tools = list(CANONICAL_TOOLS)
for t in sorted(all_tool_names):
    if t not in matrix_tools:
        matrix_tools.append(t)

n = len(matrix_tools)
matrix_data = [[0] * n for _ in range(n)]
for (tA, tB), cnt in tool_conflict_matrix.items():
    if tA in matrix_tools and tB in matrix_tools:
        iA, iB = matrix_tools.index(tA), matrix_tools.index(tB)
        matrix_data[iA][iB] = cnt
        matrix_data[iB][iA] = cnt

short = [SHORT_NAMES.get(t, t[:12]) for t in matrix_tools]

df_matrix = pd.DataFrame(matrix_data, index=short, columns=short)
df_matrix.index.name = "Tool ↓ / Tool →"

def style_heatmap(df):
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    max_val = max(df.values.max(), 1)
    for i in range(len(df)):
        for j in range(len(df.columns)):
            v = df.iloc[i, j]
            if i == j:
                styles.iloc[i, j] = "background-color: #e8e8e8; color: #aaa"
            elif v > 0:
                intensity = min(v / max_val, 1.0)
                r = int(255 - intensity * 70)
                g = int(255 - intensity * 120)
                b = int(255 - intensity * 30)
                styles.iloc[i, j] = f"background-color: rgb({r},{g},{b}); font-weight: bold"
            else:
                styles.iloc[i, j] = "color: #ccc"
    return styles

styled_matrix = (df_matrix
    .style
    .apply(style_heatmap, axis=None)
    .set_caption("Conflicts between each tool pair (symmetric)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "14px"), ("font-weight", "bold")]},
        {"selector": "td", "props": [("text-align", "center"), ("min-width", "60px"), ("padding", "6px")]},
        {"selector": "th", "props": [("text-align", "center"), ("padding", "6px")]},
    ]))
display(styled_matrix)

# ═══════════════════════════════════════════════════════════════════
# TABLE 2 — COMBINED CONFLICT SUMMARY
# ═══════════════════════════════════════════════════════════════════
total_det = sum(det_counts.values())
total_res = sum(res_counts.values())
total_abs = sum(abstain_by_reason.values())

print("\n" + "=" * 90)
print("  CONFLICT RESOLUTION SUMMARY")
print("=" * 90)

summary_rows = [
    {"Metric": "Total Conflicts Detected",    "Value": total_det},
    {"Metric": "Total Conflicts Resolved",    "Value": total_res},
    {"Metric": "Argumentation Cycles",         "Value": cycle_count},
    {"Metric": "Total Abstentions",            "Value": total_abs},
    {"Metric": "Human-Review Flagged",         "Value": human_review_count},
    {"Metric": "─── Severity ───",             "Value": ""},
    {"Metric": "  Critical",                   "Value": severity_counts.get("critical", 0)},
    {"Metric": "  Moderate",                   "Value": severity_counts.get("moderate", 0)},
    {"Metric": "  Minor",                      "Value": severity_counts.get("minor", 0)},
]
# Top abstention reasons
if abstain_by_reason:
    summary_rows.append({"Metric": "─── Abstention Reasons ───", "Value": ""})
    for reason, cnt in abstain_by_reason.most_common(5):
        summary_rows.append({"Metric": f"  {reason}", "Value": cnt})

df_summary = pd.DataFrame(summary_rows)
styled_summary = (df_summary
    .style
    .hide(axis="index")
    .set_caption("Detection → Resolution → Abstention Pipeline")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold")]},
        {"selector": "td:nth-child(2)", "props": [("text-align", "right"), ("font-weight", "bold")]},
        {"selector": "td", "props": [("padding", "3px 10px")]},
    ]))
display(styled_summary)

# ── Per-pair detail (only pairs with conflicts) ──
if tool_conflict_matrix:
    detail_rows = []
    for (tA, tB), cnt in sorted(tool_conflict_matrix.items(), key=lambda x: -x[1]):
        types = tool_conflict_by_type.get((tA, tB), {})
        sevs  = tool_conflict_severity.get((tA, tB), {})
        pathologies = [(p, c) for (a, b, p), c in tool_conflict_pathology.items()
                       if (a, b) == (tA, tB)]
        top_path = sorted(pathologies, key=lambda x: -x[1])[:3]
        top_str  = ", ".join(f"{p}({c})" for p, c in top_path) if top_path else "—"

        sA = SHORT_NAMES.get(tA, tA[:12])
        sB = SHORT_NAMES.get(tB, tB[:12])
        detail_rows.append({
            "Pair": f"{sA} vs {sB}",
            "Total": cnt,
            "Semantic": types.get("semantic", 0),
            "Presence": types.get("presence", 0),
            "Critical": sevs.get("critical", 0),
            "Moderate": sevs.get("moderate", 0),
            "Minor": sevs.get("minor", 0),
            "Top Pathologies": top_str,
        })

    df_detail = pd.DataFrame(detail_rows)
    styled_detail = (df_detail
        .style
        .background_gradient(subset=["Total"], cmap="OrRd", vmin=0)
        .hide(axis="index")
        .set_caption("Per-Pair Conflict Breakdown")
        .set_table_styles([
            {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold")]},
            {"selector": "td", "props": [("padding", "4px 8px")]},
        ]))
    display(styled_detail)

# ── Save JSON ──
all_tables = {
    "tool_matrix": {
        "tools": matrix_tools,
        "matrix": {f"{a} vs {b}": c for (a, b), c in tool_conflict_matrix.items()},
        "by_type": {f"{a} vs {b}": dict(v) for (a, b), v in tool_conflict_by_type.items()},
        "by_severity": {f"{a} vs {b}": dict(v) for (a, b), v in tool_conflict_severity.items()},
    },
    "summary": {
        "detected": total_det, "resolved": total_res,
        "cycles": cycle_count, "abstentions": dict(abstain_by_reason),
        "severity": dict(severity_counts),
    },
}
with open(ACL_DIR / "table_conflict_resolution.json", "w") as f:
    json.dump(all_tables, f, indent=2, default=str)

print(f"\n✅ Conflict analysis: {total_det} detected, {total_res} resolved, "
      f"{total_abs} abstentions — saved to {ACL_DIR}")

In [ ]:

# ===============================
# 📦 Cell 9 — Package ACL Submission ZIP
# ===============================
import zipfile

ZIP_PATH = WORK / "medrax_premium_acl_submission.zip"

config = {
    "system": "MedRAX Premium",
    "benchmark": "ChestAgentBench",
    "questions_answered": total_ans,
    "overall_accuracy": premium.get("Overall"),
    "llm": model_name,
    "tools": [t.name for t in ALL_TOOLS],
    "conflict_resolution": {
        "enabled": True, "sensitivity": 0.25, "deferral": 0.6,
        "components": ["BERT NLI", "Rule-based", "GACL",
                       "Argumentation Graph", "Tool Trust", "Abstention Logic"],
    },
    "timestamp": datetime.now().isoformat(),
}
with open(ACL_DIR / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in ["table_accuracy_comparison.json", "table_conflict_resolution.json",
                 "experiment_config.json", "raw_results.json",
                 "cross_verification_detail.json"]:
        fp = ACL_DIR / name
        if fp.exists():
            zf.write(fp, f"tables/{name}")
            print(f"  + tables/{name}")
    for name in ["cross_verification.csv", "paper_table_accuracy.csv",
                 "error_analysis.csv"]:
        fp = ACL_DIR / name
        if fp.exists():
            zf.write(fp, f"tables/{name}")
            print(f"  + tables/{name}")
    for name in ["fig_category_comparison.pdf", "fig_category_comparison.png",
                 "fig_overall_accuracy.pdf", "fig_overall_accuracy.png"]:
        fp = ACL_DIR / name
        if fp.exists():
            zf.write(fp, f"figures/{name}")
            print(f"  + figures/{name}")
    if PREMIUM_LOG.exists():
        zf.write(PREMIUM_LOG, "logs/premium_benchmark.jsonl")
        print(f"  + logs/premium_benchmark.jsonl")

print(f"\n✅ ZIP: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**2:.1f} MB)")
